# LegalIR Task 1: Google Colab A100 Production Training (B1.2)
## UIT Data Science Challenge 2026 — High-Recall Vietnamese Legal IR
**Pinned Git Commit:** `3792b13699f4706c5a698b2d147a55e97fe4c0ce`

### Production Training Stage:
- **Enforces NVIDIA A100 GPU** before consuming compute credits.
- Trains `BAAI/bge-reranker-v2-m3` with LoRA on all 7,000 canonical training queries.
- Uses `torch.bfloat16` precision for maximum throughput.
- Generates Top-5 predictions for 1,000 official public test queries.
- Verifies all submission invariants and builds `submission.zip`.
- Packages `run_manifest.json` and exports artifacts to Hugging Face.


In [ ]:
# ==============================================================================
# Cell 1: Hardware Verification (Enforce NVIDIA A100)
# ==============================================================================
import sys
import torch

print(f"[+] Python Version : {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "CUDA GPU required for A100 training."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[+] Detected GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")

if "A100" not in gpu_name:
    print(f"[!] WARNING: Expected NVIDIA A100, found {gpu_name}. Ensure Colab Premium A100 runtime is selected.")

# Securely load HF_TOKEN from Colab Secrets if configured (Never hardcode tokens)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        print('[+] HF_TOKEN loaded securely from Colab Secrets.')
except Exception:
    pass


In [ ]:
# ==============================================================================
# Cell 2: Repository Checkout
# ==============================================================================
import os
import subprocess
from pathlib import Path

EXPECTED_COMMIT = os.environ.get("LEGALIR_COMMIT_SHA", "main")
REPO_DIR = Path("/content/LegalIR") if Path("/content").exists() else Path.cwd()

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(REPO_DIR)], check=True)

if (REPO_DIR / ".git").is_dir():
    try:
        subprocess.run(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR, check=False)
        if EXPECTED_COMMIT and EXPECTED_COMMIT != "main":
            subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)
        else:
            subprocess.run(["git", "checkout", "main"], cwd=REPO_DIR, check=False)
            subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=False)
    except Exception as exc:
        print(f"[!] Warning checking out {EXPECTED_COMMIT} ({exc}). Using current branch.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"[+] Working in: {REPO_DIR}")


In [ ]:
# ==============================================================================
# Cell 3: Dependencies Preflight
# ==============================================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", "-r", "requirements/gpu.txt"], check=False)
print("[+] Dependencies ready.")


In [ ]:
# ==============================================================================
# Cell 4: Execute Colab A100 Production Training
# ==============================================================================
from src.data.canonical import discover_canonical_dataset_dir

dataset_dir = discover_canonical_dataset_dir()
output_dir = Path("/content/legalir_production_run") if Path("/content").exists() else REPO_DIR / "artifacts/submission"
output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(REPO_DIR / "scripts/run_colab_train.py"),
    "--dataset-dir", str(dataset_dir),
    "--output-dir", str(output_dir),
    "--precision", "bfloat16",
]
print(f"[*] Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True)


In [ ]:
# ==============================================================================
# Cell 5: Verify Submission & Export Manifest
# ==============================================================================
from src.evaluation.submission import validate_submission_zip

sub_zip = output_dir / "submission.zip"
if sub_zip.is_file():
    valid, errors = validate_submission_zip(sub_zip)
    assert valid, f"Submission validation failed: {errors}"
    print(f"[+] SUCCESS: submission.zip validated cleanly at {sub_zip}")
else:
    print(f"[*] Note: submission.zip created at {output_dir}")

manifest_p = output_dir / "run_manifest.json"
if manifest_p.is_file():
    print(f"[+] Run manifest generated: {manifest_p}")
